# Quickstart: Exploring Reasoning in Language Models

This notebook demonstrates the basic usage of the interpretability framework to explore reasoning inside language models.

## What You'll Learn

1. Loading a reasoning-capable model with platform detection
2. Running inference and extracting activations
3. Inspecting model internals
4. Using the activation cache

**Estimated time**: 10 minutes  
**Hardware**: Works on CPU, MPS (Apple Silicon), or CUDA  
**Memory**: ~3-4GB RAM for 1.5B model

## Setup

In [ ]:
# Add parent directory to path if running from notebooks/
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import torch
from interpretability import load_model, ModelWrapper
from loguru import logger

# Set logging level
logger.remove()
logger.add(sys.stderr, level="INFO")

print("✓ Imports successful")

## 1. Load a Model

We'll load DeepSeek-R1-Distill-1.5B. The framework automatically detects your platform and uses the best settings:
- **macOS**: Uses MPS acceleration (if available) without quantization
- **Linux/Windows with GPU**: Can use 4-bit quantization with bitsandbytes
- **CPU only**: Full precision (slower but works everywhere)

In [ ]:
# Check platform and load model with appropriate settings
from interpretability import get_platform_info, get_recommended_device

info = get_platform_info()
print(f"Platform: {info['os']} ({info['architecture']})")
print(f"MPS available: {info['mps_available']}")
print(f"CUDA available: {info['cuda_available']}")
print(f"bitsandbytes available: {info['bitsandbytes_available']}")

# Get recommended device
device = get_recommended_device()
print(f"\nUsing device: {device}")

# Load model (quantization=None for macOS, or omit to auto-detect)
model = load_model(
    "deepseek-1.5b",  # Alias for deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
    device=device,
    quantization=None  # Use None for macOS, auto-detected otherwise
)

print(model)

## 2. Inspect Model Architecture

Let's explore the model's internal structure.

In [ ]:
# Model metadata
print(f"Number of layers: {model.get_num_layers()}")
print(f"Hidden dimension: {model.get_hidden_dim()}")
print(f"Attention heads (layer 0): {model.get_attention_heads(0)}")
print(f"Device: {model.device}")

# Memory statistics
mem_stats = model.memory_stats()
print(f"\nMemory Usage:")
print(f"  Parameters: {mem_stats['num_parameters_m']:.1f}M")
print(f"  Size: {mem_stats['param_size_mb']:.1f} MB")
print(f"  Quantization: {mem_stats['quantization']}")

## 3. Run Inference

Let's run a simple reasoning task and extract internal activations.

In [ ]:
# Prepare a reasoning prompt
prompt = "What is 17 * 23? Let me calculate step by step."

# Tokenize and move to device (important for MPS/CUDA)
inputs = model.tokenize(prompt, move_to_device=True)
print(f"Input shape: {inputs['input_ids'].shape}")
print(f"Input device: {inputs['input_ids'].device}")
print(f"Tokens: {model.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")

In [ ]:
# Run forward pass with attention and hidden state extraction
with torch.no_grad():
    output = model.forward(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        output_hidden_states=True,
        output_attentions=True
    )

print(f"\nOutput logits shape: {output.logits.shape}")
print(f"Number of hidden state layers: {len(output.hidden_states)}")
print(f"Number of attention layers: {len(output.attentions)}")
print(f"\nHidden state shape (layer 0): {output.hidden_states[0].shape}")
print(f"Attention shape (layer 0): {output.attentions[0].shape}")

## 4. Using Hooks for Activation Extraction

The hook manager allows us to intercept activations at specific layers.

In [ ]:
# List available modules
attention_modules = model.hook_manager.list_modules(pattern="self_attn")
print(f"Found {len(attention_modules)} attention modules")
print(f"Example modules: {attention_modules[:3]}")

In [ ]:
# Register hooks on specific layers using context manager
layer_names = [f"model.layers.{i}" for i in [0, 10, 20]]

with model.hook_manager.temporary_hooks(layer_names) as activations:
    # Run inference
    output = model.forward(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )
    
    # Activations are captured
    print(f"\nCaptured activations from {len(activations)} layers")
    
    for layer_name, acts in list(activations.items())[:3]:
        print(f"{layer_name}: {len(acts)} tensors, shape {acts[0].shape}")

## 5. Activation Caching

Cache activations for later analysis.

In [ ]:
# Access the model's cache
cache = model.cache

# Get hidden states (run forward pass if needed)
# Note: We need to re-run since the hook example above overwrote 'output'
with torch.no_grad():
    output_for_cache = model.forward(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        output_hidden_states=True
    )

# Store hidden states from our run
for layer_idx, hidden_state in enumerate(output_for_cache.hidden_states):
    cache.store(
        key=f"prompt1_layer{layer_idx}",
        activations=hidden_state
    )

print(f"Stored {len(output_for_cache.hidden_states)} layers to cache")
print(f"\nCache statistics:")
stats = cache.stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

In [ ]:
# Retrieve cached activations
layer_10_acts = cache.retrieve("prompt1_layer10")
print(f"\nRetrieved layer 10 activations: {layer_10_acts.shape}")

## 6. Generate Text with Reasoning

Let's see the model generate a complete response.

In [ ]:
# Generate text
prompt = "What is the square root of 144? Let me think."
inputs = model.tokenize(prompt, move_to_device=True)

with torch.no_grad():
    generated = model.generate(
        inputs["input_ids"],
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

# Decode and print
output_text = model.decode(generated[0])
print(f"Generated text:\n{output_text}")

## Summary

In this notebook, you learned how to:

1. ✅ Detect your platform and load model with appropriate settings
2. ✅ Inspect model architecture and memory usage
3. ✅ Run forward passes and extract attention/hidden states
4. ✅ Use hooks to capture activations at specific layers
5. ✅ Cache activations for later analysis
6. ✅ Generate text with the model

## Platform-Specific Notes

- **macOS**: Uses MPS acceleration without quantization (bitsandbytes not supported)
- **Linux/Windows with GPU**: Can use 4-bit quantization for memory efficiency
- **CPU only**: Full precision, slower but works everywhere

## Next Steps

- **Notebook 02**: Attention pattern analysis
- **Notebook 03**: Circuit discovery with activation patching
- **Notebook 04**: Logit lens and answer emergence
- **Notebook 05**: Comparative analysis across prompts

## Troubleshooting

If you encounter issues:
- **MPS errors**: Make sure `move_to_device=True` when tokenizing
- **Memory errors**: Use a smaller model or CPU device
- **Import errors**: Run `pip install -e .` from project root

See [MACOS_COMPLETE_FIX.md](../MACOS_COMPLETE_FIX.md) for comprehensive troubleshooting.

---

**Phase 1 Complete!** 🎉

You now have the foundation to explore reasoning inside language models.